# 02 - Random Topology Churn

Plan section 104: the mutation engine works, cost is measurable, and the random control is reproducible.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Random topology churn (plan section 104)

Proves the second notebook's claims: the topology mutation engine
works, mutation cost is measurable, and the random control is
reproducible. Mutations are applied directly through the
prune/grow operators on a real genome, then rebuilt
weight-preserving into a live model.


In [ ]:
from act.model import model_from_genome
from act.topology import (
    create_genome,
    feasible_expert_mutations,
    genome_hash,
    grow_expert,
    prune_expert,
)

genome = create_genome(42, hidden_size=24, vocab_size=32, depth=1,
                       ffn='moe', expert_count=4, active_experts=2)
model = model_from_genome(genome, context_size=12)
hash_before = genome_hash(genome)
print('feasible mutations:', feasible_expert_mutations(genome)[:4], '...')

In [ ]:
# Grow one expert: the operator returns the new expert's ID, the
# genome hash changes, and the layer gains exactly one expert.
layer = genome.layers[0]
count_before = len(layer.experts)
added_id = grow_expert(genome, layer.id, step=12)
assert added_id is not None, 'grow was not feasible on a fresh genome'
assert added_id == f"{layer.id}E{count_before}"
assert genome_hash(genome) != hash_before
assert len(genome.layers[0].experts) == count_before + 1
print('grew', added_id, '->', len(genome.layers[0].experts), 'experts')

In [ ]:
# Rebuild the mutated model: by-expert-ID weight transfer must retain
# every existing expert's weights and report more parameters.
import torch
from act.model import rebuild_model_from_genome

params_before = model.parameter_count
grown_model = rebuild_model_from_genome(model, genome, context_size=12)
assert grown_model.parameter_count > params_before
retained = grown_model.state_dict()[
    'blocks.0.experts.0.0.weight']
source = model.state_dict()['blocks.0.experts.0.0.weight']
assert torch.equal(retained, source), 'expert weights did not survive'
model = grown_model
print('rebuilt:', model.parameter_count, 'parameters (was', params_before, ')')

In [ ]:
# Prune the added expert back and verify the hash round trip.
assert prune_expert(genome, layer.id, added_id)
assert genome_hash(genome) == hash_before, 'prune did not restore the genome'
print('prune restored hash', hash_before)

In [ ]:
# Mutation cost is measurable: every mutation rebuilds the model and
# books parameter_count x 4 rebuild FLOPs into search, while the fixed
# arm books none. The registry round trip: one experiment, one record.
from act.evolution.experiment import ExperimentConfig, run_experiment

result = run_experiment(ExperimentConfig(
    seed=42, steps=48, arms='fixed,random', device='cpu',
))
random_arm = next(arm for arm in result.arms if arm.mode == 'random')
fixed_arm = next(arm for arm in result.arms if arm.mode == 'fixed')
assert random_arm.mutation_count == 3
assert fixed_arm.mutation_count == 0 and fixed_arm.search_flops == 0
assert random_arm.search_flops > 0  # 3 rebuilds booked
assert len(random_arm.mutation_history) == 3
print('random arm:', random_arm.mutation_count, 'mutations,',
      random_arm.training_flops, 'training FLOPs,',
      random_arm.search_flops, 'search FLOPs')

In [ ]:
# Registry round trip: one experiment appends exactly one JSONL record.
import json
from act.evolution.registry import append_result

registry = 'artifacts/notebook-records/experiments.jsonl'
record = append_result(registry, result)
lines = Path(registry).read_text().strip().splitlines()
assert lines[-1] == json.dumps(record, sort_keys=True)
print('appended', record['experiment_id'], 'to', registry)

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '02_random_topology_churn',
 'mutations': random_arm.mutation_count,
 'training_flops': random_arm.training_flops,
 'architecture_hash': random_arm.architecture_hash,
 'registry_round_trip': True}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '02-random-churn.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)